# LangGraph + LangChain
* 15.1에서 만든 **Node · Graph**와 15.2의 **`add_messages` · Conditional Edge**에 LangChain LLM을 연결합니다.

---
* `day15` Conda 환경을 사용합니다.
* LangChain OpenAI 연동 패키지를 추가로 설치합니다.

In [1]:
!pip install langchain-openai

  Using cached tiktoken-0.13.0-cp312-cp312-win_amd64.whl.metadata (6.8 kB)
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 17.5 MB/s  0:00:00
Using cached tiktoken-0.13.0-cp312-cp312-win_amd64.whl (874 kB)
   ---------------------------------------- 0.0/676.6 kB ? eta -:--:--
   ---------------------------------------- 676.6/676.6 kB 25.9 MB/s  0:00:00

   ------ --------------------------------- 1/6 [regex]
   -------------------------- ------------- 4/6 [openai]
   -------------------------- ------------- 4/6 [openai]
   -------------------------- ------------- 4/6 [openai]
   -------------------------- ------------- 4/6 [openai]
   -------------------------- ------------- 4/6 [openai]
   -------------------------- ------------- 4/6 [openai]
   -------------------------- ------------- 4/6 [openai]
   -------------------------- ------------- 4/6 [openai]
   -------------------------- ------------- 4/6 [openai]

In [2]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
print('WORKDIR :', WORKDIR)

WORKDIR : c:\Users\finey\OneDrive\Desktop\실습폴더\15일차


---
## Node 안에서 LangChain LLM 호출

Node 안에서 **`ChatOpenAI.invoke()`** 로 실제 LLM 응답을 받습니다.

| Message 타입 | 역할 |
|----------------|------|
| `HumanMessage` | 사용자 메시지 |
| `AIMessage` | 모델 응답 메시지 |
| `SystemMessage` | 시스템 프롬프트 (역할·규칙 지정) |

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7, api_key=OPENAI_API_KEY)

# 그래프 밖에서 먼저 단독 호출해 봅니다
demo_reply = llm.invoke([
    SystemMessage(content='한 문장으로만 답하세요.'),
    HumanMessage(content='LangGraph가 뭐야?'),
])
demo_reply.content

'LangGraph는 자연어 처리와 관련된 다양한 작업을 지원하는 그래프 기반의 데이터 구조나 프레임워크를 의미할 수 있습니다.'

* Node 내부에서 LangChain 기반 LLM을 실행해 보겠습니다

In [4]:
from pydantic import BaseModel

class ChatState(BaseModel):
    question: str = ''
    answer: str = ''

In [5]:
def chat_node(state: ChatState) -> dict:
    """LangChain LLM을 Node 안에서 1회 호출합니다."""
    response = llm.invoke([
        SystemMessage(content='친절한 한국어 어시스턴트입니다. 2문장 이내로 답하세요.'),
        HumanMessage(content=state.question),
    ])
    return {'answer': response.content}

In [6]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ChatState)
workflow.add_node('chat', chat_node)
workflow.add_edge(START, 'chat')
workflow.add_edge('chat', END)

single_app = workflow.compile()

In [7]:
result = single_app.invoke(ChatState(question='LangGraph의 Node는 무엇을 하나요?'))
print('질문:', result['question'])
print('답변:', result['answer'])

질문: LangGraph의 Node는 무엇을 하나요?
답변: LangGraph의 Node는 데이터의 특정 요소를 나타내며, 이 요소들 간의 관계를 정의하고 연결하는 역할을 합니다. 이를 통해 복잡한 정보를 구조화하고 효율적으로 처리할 수 있습니다.


In [8]:
for chunk in single_app.stream(ChatState(question='LangGraph에서 Edge는 무엇인가요?')):
    print(chunk)

{'chat': {'answer': 'LangGraph에서 Edge는 두 노드 간의 관계나 연결을 나타내는 요소입니다. 이는 그래프 구조에서 정보나 데이터의 흐름을 표현하는 데 사용됩니다.'}}


---
## `chat_history`에 LangChain 메시지 저장

LangChain의 `HumanMessage` / `AIMessage`를 그대로 history에 쌓고,  
다음 턴에 **전체 history를 LLM에 넘겨** 맥락을 유지합니다.

In [9]:
from typing import Annotated

from pydantic import Field
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage


class ChatState(BaseModel):
    # 15.2와 동일: add_messages → Node는 새 메시지 리스트만 반환
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    question: str = ''  # 이번 턴에 추가할 사용자 질문

In [10]:
def multi_chat_node(state: ChatState) -> dict:
    # 이번 턴 사용자 메시지를 history에 추가
    user_msg = HumanMessage(content=state.question) # -> HumanMessage: 사용자(인간)의 메시지를 저장하는 메시지 클래스
    messages_for_llm = state.chat_history + [user_msg]

    # 지금까지의 대화 전체를 LLM에 전달
    response = llm.invoke([
        SystemMessage(content='친절한 한국어 어시스턴트입니다. 이전 대화를 기억하세요.'),
        *messages_for_llm,
    ])

    # add_messages 덕분에 [user_msg, response]만 반환해도 기존 history에 이어 붙음
    return {'chat_history': [user_msg, response]}


multi_workflow = StateGraph(ChatState)
multi_workflow.add_node('chat', multi_chat_node)
multi_workflow.add_edge(START, 'chat')
multi_workflow.add_edge('chat', END)

multi_app = multi_workflow.compile()

## 이름 기억 테스트

In [13]:
state = multi_app.invoke(ChatState(question='내 이름은 민수야. 기억해줘.'))

print('--- 1턴 ---')
for msg in state['chat_history']:
    role = '사용자' if msg.type == 'human' else 'AI'
    print(f'[{role}] {msg.content}')

--- 1턴 ---
[사용자] 내 이름은 민수야. 기억해줘.
[AI] 안녕하세요, 민수님! 반갑습니다. 무엇을 도와드릴까요?


In [14]:
# 이전 state를 이어서 새 질문만 넣습니다
state['question'] = "내 이름이 뭐지?"
state = multi_app.invoke(state)

print('--- 2턴 이후 전체 history ---')
for msg in state['chat_history']:
    role = '사용자' if msg.type == 'human' else 'AI'
    print(f'[{role}] {msg.content}')

--- 2턴 이후 전체 history ---
[사용자] 내 이름은 민수야. 기억해줘.
[AI] 안녕하세요, 민수님! 반갑습니다. 무엇을 도와드릴까요?
[사용자] 내 이름이 뭐지?
[AI] 민수님이십니다! 어떻게 도와드릴까요?


### `add_messages`가 필요한 이유

add_message 속성 추가 없이 Node가 **새 메시지만** 반환하면  
기존 history가 **통째로 덮어써져** 이전 대화가 사라집니다.

In [15]:
class BrokenChatState(BaseModel):
    chat_history: list[BaseMessage] = Field(default_factory=list) # -> Annotated[, add_messages] 없음
    question: str = ''


def broken_chat_node(state: BrokenChatState) -> dict:
    user_msg = HumanMessage(content=state.question)
    response = llm.invoke([HumanMessage(content=state.question)])
    return {'chat_history': [user_msg, response]}  # 기존 history 무시 → 덮어쓰기


broken_app = StateGraph(BrokenChatState)
broken_app.add_node('chat', broken_chat_node)
broken_app.add_edge(START, 'chat')
broken_app.add_edge('chat', END)
broken_app = broken_app.compile()

s1 = broken_app.invoke(BrokenChatState(question='내 이름은 영희야.'))
s2 = broken_app.invoke(BrokenChatState(
    chat_history=s1['chat_history'],
    question='내 이름이 뭐였지?',
))

print('history 메시지 수:', len(s2['chat_history']))
print('마지막 AI 답:', s2['chat_history'][-1].content)

history 메시지 수: 2
마지막 AI 답: 죄송하지만, 당신의 이름을 알 수 있는 정보는 없습니다. 이름을 말씀해 주시면 기억해 드릴 수 있습니다!


### `input()` 기반으로 채팅하기
* LangGraph로 완전한 멀티턴 대화를 구현하려면, 대화 지속/종료를 선택하는 분기가 필요합니다
* `add_conditional_edges`와 적절한 `router`를 활용해 `input()` 기반으로 채팅을 구현해 보세요

In [16]:
from typing import Annotated

from pydantic import Field
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage


class ChatState(BaseModel):
    # 15.2와 동일: add_messages → Node는 새 메시지 리스트만 반환
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    question: str = ''  # 이번 턴에 추가할 사용자 질문

def chat_node(state: ChatState) -> dict:
    # 이번 턴 사용자 메시지를 history에 추가
    user_msg = HumanMessage(content=state.question) # -> HumanMessage: 사용자(인간)의 메시지를 저장하는 메시지 클래스
    messages_for_llm = state.chat_history + [user_msg]

    # 지금까지의 대화 전체를 LLM에 전달
    response = llm.invoke([
        SystemMessage(content='You are a kind AI Assistant. You MUST answer in Korean Language.'),
        *messages_for_llm,
    ])

    # add_messages 덕분에 [user_msg, response]만 반환해도 기존 history에 이어 붙음
    return {'chat_history': [user_msg, response]}


multi_workflow = StateGraph(ChatState)
multi_workflow.add_node('chat', multi_chat_node)
multi_workflow.add_edge(START, 'chat')
multi_workflow.add_edge('chat', END)

In [ ]:
multi_app = multi_workflow.compile()

In [21]:
current_state = {
    "chat_history": [],
    "question": ""
}

while True:
    user_input = input("\n[User]: ")
    if user_input.lower() == 'q':
        break
    current_state['question'] = user_input
    current_state = multi_app.invoke(current_state)
    print(current_state['chat_history'][-1].content)

안녕하세요, 홍길동님! 만나서 반가워요. 어떻게 도와드릴까요?
홍길동님이라고 하셨죠? 맞나요?


---
## 순환 그래프 — AI끼리 대화 시켜보기

15.2의 **A ↔ B ↔ judge** 순환 구조에 LangChain LLM을 넣습니다.

| Node | 역할 |
|------|------|
| `optimist` | 낙관론자 — 주제에 찬성·긍정 |
| `skeptic` | 회의론자 — 반박·비판 |

**중단 조건** (`debate_route`):
* `turn_count >= max_turns` → `END`
* 마지막 메시지에 `'합의'` 포함 → `END`
* 그 외 → `optimist`로 돌아가 순환

In [ ]:
from typing import Literal


class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0          # optimist가 말할 때마다 +1
    max_turns: int = 3           # optimist 3번 = 총 6메시지 입력 시 종료
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'


optimist_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.8, api_key=OPENAI_API_KEY)
skeptic_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.8, api_key=OPENAI_API_KEY)

In [ ]:
def optimist_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            '당신은 낙관적인 토론자입니다. 주제에 찬성하는 입장을 말하고,'
            '상대 토론자의 주장이 있다면 반박하세요'
        )),
    ]
    if not state.chat_history:
        prompts.append(HumanMessage(content=f'토론 주제: {state.topic}')) # 첫 대화이면 토론 주제를 주고
    else:
        prompts.extend(state.chat_history) # 이전 대화가 있으면 이어서 대화

    # Q. 굳이 Prompt 작성 -> extend 방식으로 구현하는 이유는?
    # A. node 별로 System Prompt가 다르기 때문에, 각각의 llm에게 system prompt는 다르게 주고
    # 대화 history는 똑같이 줘야 하기 때문

    response = optimist_llm.invoke(prompts) # llm으로부터 응답을 받고
    response.name = 'optimist' # 응답(AIMessage 형식)의 name 필드를 채워준 다음 return
    return {
        'chat_history': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    } 


def skeptic_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            '당신은 비판적인 토론자입니다. 상대 주장을 반박하세요. 2문장 이내. '
            '더 이상 반박이 어려워 패배를 인정한다면 "합의"라고 말하십시오'
        )),
        *state.chat_history,
    ]
    response = skeptic_llm.invoke(prompts)
    response.name = 'skeptic'
    return {
        'chat_history': [response],
        'last_speaker': 'skeptic',
    }

### `route` 함수와 Conditional Edge 구현
* `debate_route`는 **다음에 갈 Node 이름** 또는 `END`를 반환합니다.
* `add_conditional_edges('skeptic', debate_route)`

In [ ]:
def debate_route(state: DebateState):
    if state.turn_count >= state.max_turns:
        return END
    last_text = state.chat_history[-1].content if state.chat_history else ''
    if '합의' in last_text:
        return END
    return 'optimist'


debate_workflow = StateGraph(DebateState)
debate_workflow.add_node('optimist', optimist_node)
debate_workflow.add_node('skeptic', skeptic_node)

debate_workflow.add_edge(START, 'optimist')
debate_workflow.add_edge('optimist', 'skeptic')
debate_workflow.add_conditional_edges('skeptic', debate_route)

debate_app = debate_workflow.compile()

In [ ]:
init_debate = DebateState(topic='AI 발전은 인간의 행복에 긍정적인 영향을 줄 것이다.')

for chunk in debate_app.stream(init_debate):
    print(chunk)

## 사회자 추가하기

토론 그래프에 **`moderator` Node**를 추가해 보세요.

* 매 라운드 끝에 양쪽 주장을 한 줄로 요약
* `debate_route`에서 `moderator`를 거친 뒤 `optimist`로 보내기
* 사회자가 '종료'를 언급해야 토론이 종료되도록 종료 조건 수정